<a href="https://colab.research.google.com/github/madhuraneeli2008/EDAV-PROJECT/blob/main/Metro_Rail_Passenger_Usage_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metro Rail Passenger Usage Analysis Using Exploratory Data Analysis (EDA)

**Course project:** Exploratory Data Analysis & Visualization (EDAV)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter/Google Colab

## Project objective
Analyze metro passenger counts to understand station demand, peak hours, daily and monthly patterns, and possible operational improvements.

> **Important data note:** This notebook does not claim to contain official Hyderabad Metro ridership data. It first lets you upload your own CSV. If you do not upload a file, it creates a clearly labelled **synthetic demonstration dataset** so you can test every section. Do not present synthetic results as real-world findings.

## Workflow
1. Load a CSV or generate synthetic demonstration data.
2. Inspect columns, data types, missing values, duplicates, and data quality.
3. Clean and validate passenger counts and dates.
4. Create useful time features.
5. Explore station, hour, weekday, and date/month patterns.
6. Create visualizations and calculate data-driven findings.
7. Export cleaned data and summary tables.

### Expected data format
The code attempts to detect common column names. The most useful columns are a date/timestamp, station name, and passenger count. Optional columns include hour, day, entries, exits, line, or train frequency.

## 1. Install and import libraries

Google Colab usually includes these libraries. Run the installation cell only if an import fails.

In [1]:
# If needed, uncomment and run:
# %pip install pandas numpy matplotlib seaborn

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load your dataset

**Option A (recommended):** In Google Colab, run this cell and upload your metro passenger CSV file.  
**Option B:** If you do not have a dataset yet, set `USE_SYNTHETIC_DEMO = True`. The generated data is fictional and only for learning/testing.

The notebook uses flexible column detection, but if your dataset uses unusual column names, edit the mapping cell below.

In [2]:
USE_SYNTHETIC_DEMO = True  # Change to False to upload your own CSV

if not USE_SYNTHETIC_DEMO:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise ValueError("No CSV file was uploaded. Please upload a .csv file.")
        DATA_PATH = csv_files[0]
        df_raw = pd.read_csv(DATA_PATH)
        DATA_SOURCE = f"Uploaded file: {DATA_PATH}"
    except ImportError:
        DATA_PATH = input("Enter the path to your CSV file (for example, metro_data.csv): ").strip()
        df_raw = pd.read_csv(DATA_PATH)
        DATA_SOURCE = f"Local file: {DATA_PATH}"
else:
    # SYNTHETIC DATA ONLY — not official metro statistics.
    rng = np.random.default_rng(42)
    dates = pd.date_range("2025-01-01", periods=120, freq="D")
    stations = ["Ameerpet", "Miyapur", "LB Nagar", "Raidurg", "Secunderabad"]
    rows = []
    for date in dates:
        for station in stations:
            for hour in [7, 8, 9, 10, 13, 17, 18, 19, 20]:
                base = {
                    "Ameerpet": 850, "Miyapur": 620, "LB Nagar": 570,
                    "Raidurg": 700, "Secunderabad": 760
                }[station]
                peak_factor = 1.8 if hour in [8, 9, 18, 19] else 0.75
                weekend_factor = 0.72 if date.dayofweek >= 5 else 1.0
                monthly_factor = 1 + 0.08 * np.sin(2 * np.pi * date.dayofyear / 365)
                passengers = max(0, int(base * peak_factor * weekend_factor * monthly_factor
                                        + rng.normal(0, 90)))
                rows.append({
                    "Date": date,
                    "Station": station,
                    "Hour": hour,
                    "Passengers": passengers
                })
    df_raw = pd.DataFrame(rows)
    DATA_SOURCE = "Synthetic demonstration data generated in this notebook"

print("Data source:", DATA_SOURCE)
print("Dataset shape:", df_raw.shape)
display(df_raw.head())

Data source: Synthetic demonstration data generated in this notebook
Dataset shape: (5400, 4)


,Date,Station,Hour,Passengers
0,2025-01-01,Ameerpet,7,665
1,2025-01-01,Ameerpet,8,1438
2,2025-01-01,Ameerpet,9,1599
3,2025-01-01,Ameerpet,10,723
4,2025-01-01,Ameerpet,13,462


## 3. Inspect the dataset

These checks show how many records and columns are present, what data types are used, and where missing values or duplicate rows occur.

In [ ]:
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nData types:")
display(df_raw.dtypes.rename("data_type").to_frame())

print("\nMissing values by column:")
display(df_raw.isna().sum().rename("missing_count").to_frame())

print("\nExact duplicate rows:", int(df_raw.duplicated().sum()))

print("\nDescriptive statistics:")
display(df_raw.describe(include="all").T)

## 4. Detect the relevant columns

Column names differ between datasets. This cell looks for common alternatives such as `station_name`, `timestamp`, `ridership`, `passenger_count`, and `entries`. If detection is incorrect, edit the selected values manually in the mapping dictionary.

In [ ]:
def normalize_col_name(name):
    return str(name).strip().lower().replace(" ", "_").replace("-", "_")

original_columns = list(df_raw.columns)
normalized_to_original = {normalize_col_name(c): c for c in original_columns}

def find_column(candidates):
    for candidate in candidates:
        if candidate in normalized_to_original:
            return normalized_to_original[candidate]
    return None

column_map = {
    "date": find_column(["date", "travel_date", "service_date", "timestamp", "datetime", "date_time"]),
    "station": find_column(["station", "station_name", "station_id", "stop", "stop_name"]),
    "passengers": find_column(["passengers", "passenger_count", "ridership", "passenger_volume",
                               "total_passengers", "count", "footfall"]),
    "hour": find_column(["hour", "hour_of_day", "travel_hour"]),
    "day": find_column(["day", "day_of_week", "weekday"]),
    "entries": find_column(["entries", "entry_count", "boardings", "in_count"]),
    "exits": find_column(["exits", "exit_count", "alightings", "out_count"]),
    "line": find_column(["line", "line_name", "metro_line", "route"])
}

print("Detected column mapping:")
for key, value in column_map.items():
    print(f"{key:12} -> {value}")

# If needed, manually edit for your dataset, for example:
# column_map["date"] = "Service Date"
# column_map["station"] = "Station Name"
# column_map["passengers"] = "Passenger Count"

if column_map["passengers"] is None:
    if column_map["entries"] is not None and column_map["exits"] is not None:
        print("\nNo total passenger column detected; will calculate entries + exits as a separate total.")
    else:
        raise ValueError(
            "Could not detect a passenger-count column. Edit column_map['passengers'] "
            "to match your CSV, or provide entries and exits columns."
        )

## 5. Clean and standardize the data

Cleaning decisions depend on what a row represents. Exact duplicate rows are removed because they repeat identical records. Negative passenger counts are flagged as invalid; they are not meaningful counts. Very high counts are not automatically removed because they may be genuine peaks.

In [ ]:
df = df_raw.copy()

# Standardize station labels if a station column exists.
if column_map["station"] is not None:
    station_col = column_map["station"]
    df[station_col] = df[station_col].astype("string").str.strip()
    df[station_col] = df[station_col].replace("", pd.NA)

# Parse dates where available.
if column_map["date"] is not None:
    date_col = column_map["date"]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Convert numeric columns carefully.
numeric_cols = [column_map[k] for k in ["passengers", "hour", "entries", "exits"]
                if column_map[k] is not None]
for col in set(numeric_cols):
    df[col] = pd.to_numeric(df[col], errors="coerce")

# If total passenger count is missing but entry and exit counts exist,
# calculate a combined movement count and label it explicitly.
if column_map["passengers"] is None:
    df["Calculated_Total_Movements"] = df[column_map["entries"]].fillna(0) + df[column_map["exits"]].fillna(0)
    column_map["passengers"] = "Calculated_Total_Movements"
    print("Created total movements = entries + exits. This is not necessarily unique passengers.")

passenger_col = column_map["passengers"]
station_col = column_map["station"]
date_col = column_map["date"]
hour_col = column_map["hour"]

before = len(df)
df = df.drop_duplicates()
print("Exact duplicate rows removed:", before - len(df))

# Drop rows with missing passenger count because core passenger analyses require it.
before = len(df)
df = df.dropna(subset=[passenger_col])
print("Rows removed for missing passenger count:", before - len(df))

# Flag negative values, then exclude them from count-based analysis.
negative_count = int((df[passenger_col] < 0).sum())
print("Negative passenger counts found:", negative_count)
df = df[df[passenger_col] >= 0].copy()

# Keep non-integer passenger values visible for review; counts are normally whole numbers.
non_integer_count = int(((df[passenger_col] % 1) != 0).sum())
print("Non-integer passenger values:", non_integer_count)

# Remove rows with invalid date only if a date is needed for a date-based analysis;
# keep the rows in df for analyses that do not require dates.
print("Rows after core cleaning:", len(df))
display(df.head())

## 6. Create useful time features

This section extracts hour, weekday, month, and year from timestamps when available. If a separate hour column exists, it is used when no hour can be extracted from the date.

In [ ]:
if date_col is not None:
    df["analysis_date"] = pd.to_datetime(df[date_col], errors="coerce")
    df["analysis_day_name"] = df["analysis_date"].dt.day_name()
    df["analysis_month"] = df["analysis_date"].dt.to_period("M").astype("string")
    df["analysis_year"] = df["analysis_date"].dt.year
    df["analysis_hour_from_date"] = df["analysis_date"].dt.hour
else:
    df["analysis_date"] = pd.NaT
    df["analysis_day_name"] = pd.NA
    df["analysis_month"] = pd.NA
    df["analysis_year"] = np.nan
    df["analysis_hour_from_date"] = np.nan

if hour_col is not None:
    df["analysis_hour"] = df[hour_col]
else:
    df["analysis_hour"] = df["analysis_hour_from_date"]

if "analysis_day_name" not in df.columns or df["analysis_day_name"].isna().all():
    if column_map["day"] is not None:
        df["analysis_day_name"] = df[column_map["day"]].astype("string")
    else:
        df["analysis_day_name"] = pd.NA

print("Available analysis features:")
print([c for c in ["analysis_date", "analysis_hour", "analysis_day_name", "analysis_month", "analysis_year"] if c in df.columns])

## 7. Summary statistics and passenger usage

The totals below are sums of the passenger-count field. Confirm what each source row represents before interpreting the sum: for example, hourly counts can be summed across hours, but entry-plus-exit totals may count the same journey twice.

In [ ]:
print("Total passenger-count sum:", int(df[passenger_col].sum()))
print("Average count per record:", round(df[passenger_col].mean(), 2))
print("Median count per record:", round(df[passenger_col].median(), 2))
print("Minimum count:", df[passenger_col].min())
print("Maximum count:", df[passenger_col].max())
print("Standard deviation:", round(df[passenger_col].std(), 2))

display(df[passenger_col].describe().to_frame("passenger_count_summary"))

## 8. Visualization 1 — Passenger-count distribution

A histogram shows how often different passenger-count ranges occur. This helps reveal the typical scale and skew of the counts.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df[passenger_col].dropna(), bins=30, edgecolor="black")
plt.title("Distribution of Metro Passenger Counts")
plt.xlabel("Passenger count per record")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 9. Visualization 2 — Station-wise passenger usage

Only shown when the dataset contains a station column. A station with a higher total may have more observations, so compare the observation coverage as well.

In [ ]:
if station_col is not None:
    station_summary = (
        df.dropna(subset=[station_col])
          .groupby(station_col)[passenger_col]
          .agg(total_passenger_count="sum", average_per_record="mean", records="count")
          .sort_values("total_passenger_count", ascending=False)
    )
    display(station_summary.head(15))

    plt.figure(figsize=(11, 6))
    station_summary["total_passenger_count"].head(15).sort_values().plot(kind="barh")
    plt.title("Top Stations by Summed Passenger Count")
    plt.xlabel("Summed passenger count")
    plt.ylabel("Station")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped station chart: no station column was detected.")

## 10. Visualization 3 — Average count by station

The average per record helps complement the total. It is still affected by how records are sampled (for example, hourly vs daily data).

In [ ]:
if station_col is not None:
    avg_station = df.dropna(subset=[station_col]).groupby(station_col)[passenger_col].mean().sort_values(ascending=False)
    plt.figure(figsize=(11, 5))
    avg_station.head(15).sort_values().plot(kind="barh")
    plt.title("Stations by Average Passenger Count per Record")
    plt.xlabel("Average passenger count per record")
    plt.ylabel("Station")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped average station chart: no station column was detected.")

## 11. Visualization 4 — Hourly passenger usage / peak hours

This chart is only meaningful when an hour field or timestamp with hour-level detail is available. If timestamps contain only dates at midnight, the extracted hour is not a real travel hour; check your data before interpreting it.

In [ ]:
valid_hour = df["analysis_hour"].notna() & df["analysis_hour"].between(0, 23)
if valid_hour.any():
    hourly_summary = df.loc[valid_hour].groupby("analysis_hour")[passenger_col].agg(["sum", "mean", "count"]).sort_index()
    display(hourly_summary)

    plt.figure(figsize=(10, 5))
    hourly_summary["sum"].plot(kind="bar")
    plt.title("Passenger Count by Hour")
    plt.xlabel("Hour of day (24-hour clock)")
    plt.ylabel("Summed passenger count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Skipped hourly chart: no valid hour information was detected.")

## 12. Visualization 5 — Daily passenger trend

Daily totals require date-level information. The trend can help reveal busy and quiet dates, but a short dataset may not represent normal seasonal patterns.

In [ ]:
valid_dates = df.dropna(subset=["analysis_date"])
if not valid_dates.empty:
    daily_summary = valid_dates.groupby(valid_dates["analysis_date"].dt.date)[passenger_col].sum()
    plt.figure(figsize=(12, 5))
    daily_summary.plot(kind="line", marker=".", linewidth=1)
    plt.title("Daily Summed Passenger Count Over Time")
    plt.xlabel("Date")
    plt.ylabel("Summed passenger count")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped daily trend: no valid date column was detected.")

## 13. Visualization 6 — Weekday comparison

Compares passenger counts by day of week where date or day-name information exists. The weekday order is fixed for readability.

In [ ]:
day_col = "analysis_day_name"
if df[day_col].notna().any():
    weekday_summary = df.groupby(day_col)[passenger_col].agg(["sum", "mean", "count"])
    weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    existing = [d for d in weekday_order if d in weekday_summary.index]
    if existing:
        weekday_summary = weekday_summary.reindex(existing)
    display(weekday_summary)

    plt.figure(figsize=(10, 5))
    weekday_summary["sum"].plot(kind="bar")
    plt.title("Passenger Count by Day of Week")
    plt.xlabel("Day")
    plt.ylabel("Summed passenger count")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped weekday chart: no weekday information was detected.")

## 14. Visualization 7 — Monthly trend

Monthly aggregation is available only when dates are available. A full year or multiple years is preferable for discussing seasonal trends.

In [ ]:
if not valid_dates.empty:
    monthly_df = valid_dates.copy()
    monthly_df["month_period"] = monthly_df["analysis_date"].dt.to_period("M")
    monthly_summary = monthly_df.groupby("month_period")[passenger_col].sum()
    display(monthly_summary.to_frame("summed_passenger_count"))

    plt.figure(figsize=(11, 5))
    monthly_summary.plot(kind="line", marker="o")
    plt.title("Monthly Summed Passenger Count")
    plt.xlabel("Month")
    plt.ylabel("Summed passenger count")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped monthly trend: no valid date information was detected.")

## 15. Visualization 8 — Station vs hour heatmap

A heatmap makes it easier to compare station demand by hour. It requires both station and valid hour information.

In [ ]:
if station_col is not None and valid_hour.any():
    heatmap_data = df.loc[valid_hour & df[station_col].notna()].pivot_table(
        index=station_col, columns="analysis_hour", values=passenger_col,
        aggfunc="sum", fill_value=0
    )
    if not heatmap_data.empty:
        plt.figure(figsize=(12, max(4, 0.45 * len(heatmap_data))))
        sns.heatmap(heatmap_data, cmap="YlOrRd", annot=(heatmap_data.size <= 100), fmt=".0f")
        plt.title("Passenger Count Heatmap by Station and Hour")
        plt.xlabel("Hour of day")
        plt.ylabel("Station")
        plt.tight_layout()
        plt.show()
    else:
        print("Heatmap skipped: no usable station-hour records.")
else:
    print("Skipped heatmap: both station and hour information are required.")

## 16. Visualization 9 — Entry and exit counts (optional)

If the dataset contains entry and exit columns, compare them. These counts may describe station movements, not unique passengers.

In [ ]:
entry_col, exit_col = column_map["entries"], column_map["exits"]
if entry_col is not None and exit_col is not None:
    entry_exit_summary = pd.DataFrame({
        "Entries": pd.to_numeric(df[entry_col], errors="coerce").sum(),
        "Exits": pd.to_numeric(df[exit_col], errors="coerce").sum()
    }, index=["Total recorded movements"])
    display(entry_exit_summary)

    entry_exit_summary.T.plot(kind="bar", legend=False, figsize=(7, 4))
    plt.title("Total Recorded Entries vs Exits")
    plt.xlabel("Movement type")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Entry/exit comparison skipped: entry and exit columns were not both detected.")

## 17. Data-driven findings

This cell calculates findings from the dataset currently loaded. If synthetic data is being used, these are findings about the synthetic demonstration only—not about an actual metro system.

In [ ]:
print("=" * 65)
print("DATA-DRIVEN PROJECT SUMMARY")
print("=" * 65)
print("Data source:", DATA_SOURCE)
print("Records after core cleaning:", len(df))
print("Stations available:", df[station_col].nunique() if station_col is not None else "Not available")
print("Date range:",
      f"{df['analysis_date'].min().date()} to {df['analysis_date'].max().date()}"
      if df["analysis_date"].notna().any() else "Not available")
print("Total passenger-count sum:", round(df[passenger_col].sum(), 2))
print("Average count per record:", round(df[passenger_col].mean(), 2))

if station_col is not None and df[station_col].notna().any():
    st_totals = df.dropna(subset=[station_col]).groupby(station_col)[passenger_col].sum().sort_values(ascending=False)
    print("Highest summed station count:", st_totals.index[0], "—", round(st_totals.iloc[0], 2))
    print("Lowest summed station count:", st_totals.index[-1], "—", round(st_totals.iloc[-1], 2))

if valid_hour.any():
    h_totals = df.loc[valid_hour].groupby("analysis_hour")[passenger_col].sum().sort_values(ascending=False)
    print("Highest summed hour:", int(h_totals.index[0]), ":00 —", round(h_totals.iloc[0], 2))

if not valid_dates.empty:
    daily_counts = valid_dates.groupby(valid_dates["analysis_date"].dt.date)[passenger_col].sum()
    print("Highest daily total:", str(daily_counts.idxmax()), "—", round(daily_counts.max(), 2))
    print("Lowest daily total:", str(daily_counts.idxmin()), "—", round(daily_counts.min(), 2))

print("\nInterpretation reminder: a higher sum can reflect more observations, longer coverage, or larger station size.")
print("These associations are descriptive and do not establish cause and effect.")

## 18. Operational recommendations framework

Use the actual results above to complete your final report. These are possible actions to consider, not claims that the dataset proves a specific problem.

- **Peak-hour scheduling:** If hourly counts consistently peak at certain times, operators can evaluate whether additional service is needed.
- **Station crowd management:** If a station has consistently high demand, review passenger flow, signage, staffing, and platform management.
- **Resource planning:** If demand varies by weekday or month, staffing and maintenance planning may be adjusted accordingly.
- **Data quality improvement:** Consistent timestamps, station IDs, and clear definitions of passenger counts improve future analysis.

Do not recommend a service change from a short or incomplete sample alone. Consider safety, train capacity, operational constraints, and additional evidence.

## 19. Export cleaned data and summary tables

Files will be saved in the current notebook environment. In Google Colab, use the file browser to download them, or run the optional download cell below.

In [ ]:
df.to_csv("metro_cleaned_data.csv", index=False)
print("Saved: metro_cleaned_data.csv")

if station_col is not None:
    export_station_summary = (
        df.dropna(subset=[station_col])
          .groupby(station_col)[passenger_col]
          .agg(total_passenger_count="sum", average_per_record="mean", records="count")
          .sort_values("total_passenger_count", ascending=False)
    )
    export_station_summary.to_csv("station_usage_summary.csv")
    print("Saved: station_usage_summary.csv")

if not valid_dates.empty:
    export_daily = valid_dates.groupby(valid_dates["analysis_date"].dt.date)[passenger_col].sum().rename("passenger_count")
    export_daily.to_csv("daily_usage_summary.csv")
    print("Saved: daily_usage_summary.csv")

## 20. Conclusion template

After running the notebook with your real dataset, replace the bracketed parts below with calculated values.

> This project analyzed **[number of records]** metro passenger records covering **[date range]**. The analysis identified **[highest-demand station, if available]** as the station with the highest summed passenger count and **[peak hour, if available]** as the busiest recorded hour. Daily, weekday, and monthly visualizations showed **[describe observed patterns]**. These findings can support further investigation into train scheduling and station management. Results are limited by **[coverage, missing variables, sampling, or data quality limitations]** and should not be interpreted as proof of causation.

If you used the synthetic demonstration data, state that clearly and do not present the results as official ridership findings.

## 21. Limitations and future scope

### Limitations
- Results depend on the completeness, coverage, and accuracy of the dataset.
- Some datasets record entries, exits, or hourly counts rather than unique passengers.
- A short observation period may not represent holidays, special events, or seasonal variation.
- Station totals are not directly comparable if stations have different numbers of observations.
- Correlation and descriptive patterns do not prove cause and effect.

### Future scope
- Use official ridership data over a longer period.
- Add holiday, weather, event, train-frequency, and service-disruption data.
- Build an interactive dashboard using Plotly or Streamlit.
- If enough reliable historical data is available, explore time-series forecasting with proper validation.

## 22. How to run in Google Colab

1. Download this notebook file.
2. Open [Google Colab](https://colab.research.google.com/).
3. Choose **File → Upload notebook** and select this `.ipynb` file.
4. Run cells from top to bottom.
5. To analyze real data, set `USE_SYNTHETIC_DEMO = False` in Section 2 and upload your CSV.
6. After running, choose **File → Save a copy in Drive** to create your own Google Drive/Colab copy and share its link.

## References
- Pandas documentation: https://pandas.pydata.org/docs/
- NumPy documentation: https://numpy.org/doc/
- Matplotlib documentation: https://matplotlib.org/stable/
- Seaborn documentation: https://seaborn.pydata.org/
- Google Colab: https://colab.research.google.com/